# make up publication year and filter nan

In [ ]:
import pandas as pd
all_df = pd.read_csv('/itf-fi-ml/shared/users/ziyuzh/svm/data/disgent_2020/disgenet_all_annotation.csv',sep=';')
all_df.head(3)
print(len(all_df))
all_df = all_df[~(all_df['uniport'] == '[]')]
print(len(all_df))

8737
8305


In [4]:
non_pub = all_df[all_df['first_pub_year'].isna()]

In [5]:
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests
def get_earliest_pubmed_pmid_and_year(gene, disease):
    # Construct the search query
    query = f"{gene} AND {disease}"
    
    # PubMed URL to get the total number of results
    search_url = f"https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=pubmed&term={query}&retmode=json&sort=pub+date"
    
    try:
        # Fetch search results to get the total count
        response = requests.get(search_url)
        response.raise_for_status()
        total_results = int(response.json().get("esearchresult", {}).get("count", 0))

        if total_results == 0:
            print("No results found for the given query.")
            return None, None

        # Fetch only the last result (earliest publication)
        earliest_url = f"https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=pubmed&term={query}&retstart={total_results - 1}&retmax=1&retmode=json&sort=pub+date"
        earliest_response = requests.get(earliest_url)
        earliest_response.raise_for_status()
        pmid = earliest_response.json().get("esearchresult", {}).get("idlist", [])[0]

        # Fetch the publication year for the earliest publication
        summary_url = f"https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?db=pubmed&id={pmid}&retmode=json"
        summary_response = requests.get(summary_url)
        summary_response.raise_for_status()
        pub_year = summary_response.json()["result"][pmid]["pubdate"].split()[0]  # Extract the year

        return pmid, pub_year

    except requests.exceptions.RequestException as e:
        print(f"An error occurred: {e}")
        return None, None

# Define a function that will work with a row
def fetch_pubmed_data(row):
    pmid, pub_year = get_earliest_pubmed_pmid_and_year(row['gene_id'], row['disease_name'])
    return pd.Series(pub_year)

# Apply the function to each row
non_pub['first_pub_year'] = non_pub.apply(fetch_pubmed_data, axis=1)


No results found for the given query.
No results found for the given query.
No results found for the given query.
No results found for the given query.
No results found for the given query.
No results found for the given query.
No results found for the given query.
No results found for the given query.
No results found for the given query.
No results found for the given query.
No results found for the given query.
No results found for the given query.
No results found for the given query.
No results found for the given query.
No results found for the given query.
No results found for the given query.
No results found for the given query.
No results found for the given query.
No results found for the given query.
No results found for the given query.
No results found for the given query.
No results found for the given query.
No results found for the given query.
No results found for the given query.
No results found for the given query.
No results found for the given query.
No results f

/tmp/ipykernel_2892162/2018367235.py:46: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  non_pub['first_pub_year'] = non_pub.apply(fetch_pubmed_data, axis=1)


In [9]:
# filter out missing lines
len(non_pub),len(non_pub[non_pub['first_pub_year'].isna()]),len(non_pub[~(non_pub['first_pub_year'].isna())])

(3120, 413, 2707)

In [8]:
all_df_ori = all_df[~(all_df['first_pub_year'].isna())]
len(all_df_ori)

5185

In [10]:
dga_combined = pd.concat([all_df_ori, non_pub[~(non_pub['first_pub_year'].isna())]], ignore_index=True)

In [12]:
dga_combined.to_csv('/itf-fi-ml/shared/users/ziyuzh/svm/data/disgent_2020/dga_all_pub.csv',index=False)

# generta ppi emb from two versions

In [11]:
import os
import pickle
import numpy as np
import pandas as pd

local_stringdb = os.path.join('/itf-fi-ml/shared/users/ziyuzh/svm/data/stringdb/2023')

ppidf = pd.read_csv(os.path.join(local_stringdb,'9606.protein.info.v12.0.txt'), sep='\t', header=0, usecols=['#string_protein_id', 'preferred_name'])
ppidf['preferred_name'] = ppidf['preferred_name'].str.upper()
stringId2name = ppidf.set_index('#string_protein_id')['preferred_name'].to_dict()
name2stringId = ppidf.set_index('preferred_name')['#string_protein_id'].to_dict()
ppidf = pd.read_csv(os.path.join(local_stringdb,'9606.protein.aliases.v12.0.txt'), sep='\t', header=0, usecols=['#string_protein_id', 'alias']).drop_duplicates(['alias'], keep='first')
ppidf['alias'] = ppidf['alias'].str.upper()
aliases2stringId = ppidf.set_index('alias')['#string_protein_id'].to_dict()

year = '2016'
full = True
local_stringdb = os.path.join('/itf-fi-ml/shared/users/ziyuzh/svm/data/stringdb',year)
if full:
    # ppi_connection = pd.read_csv(os.path.join(local_stringdb,'9606.protein.links.v12.0.txt'), sep=' ', header=0).convert_dtypes().replace(0, float('nan'))
    ppi_connection = pd.read_csv(os.path.join(local_stringdb,'9606.protein.links.v10.txt'), sep=' ', header=0).convert_dtypes().replace(0, float('nan'))
    # ppi_connection = pd.read_csv(os.path.join(local_stringdb,'9606.protein.links.v10.5.txt'), sep=' ', header=0).convert_dtypes().replace(0, float('nan'))

    # ppi_connection = pd.read_csv(os.path.join(local_stringdb,'9606.protein.links.v9.1.txt'), sep=' ', header=0).convert_dtypes().replace(0, float('nan'))

else:
    ppi_connection = pd.read_csv(os.path.join(local_stringdb,'9606.protein.physical.links.full.v12.0.txt'), sep=' ', header=0).convert_dtypes().replace(0, float('nan'))

ppi_connection[['protein1', 'protein2']] = np.sort(ppi_connection[['protein1', 'protein2']], axis=1)
ppi_connection = ppi_connection.drop_duplicates()
ppi_connection

,protein1,protein2,combined_score
0,9606.ENSP00000000233,9606.ENSP00000003084,150
1,9606.ENSP00000000233,9606.ENSP00000003100,215
2,9606.ENSP00000000233,9606.ENSP00000005257,223
3,9606.ENSP00000000233,9606.ENSP00000005340,193
4,9606.ENSP00000000233,9606.ENSP00000006101,415
...,...,...,...
8537757,9606.ENSP00000472847,9606.ENSP00000473036,188
8537758,9606.ENSP00000472847,9606.ENSP00000473172,281
8538703,9606.ENSP00000472867,9606.ENSP00000472929,150
8540618,9606.ENSP00000472929,9606.ENSP00000473233,158


In [12]:
proteins = set(set(ppi_connection['protein1'].unique()) | set(ppi_connection['protein2'].unique()))
len(proteins)

19247

In [13]:
nodes_to_keep = set(proteins) & set(stringId2name.keys())
len(nodes_to_keep)

14298

In [14]:
ppi_connection = ppi_connection[ppi_connection['protein1'].isin(nodes_to_keep)]
ppi_connection = ppi_connection[ppi_connection['protein2'].isin(nodes_to_keep)]
len(ppi_connection)

2389568

In [15]:
ppi_connection[['protein1', 'protein2']].to_csv('/itf-fi-ml/shared/users/ziyuzh/svm/data/ppi_clean_2017.txt', sep='\t', index=False, header=False)

In [16]:
### PPI features from node2vec
from pecanpy import pecanpy as node2vec

def Runnode2vec(filepath):
    n2v = node2vec.SparseOTF(p=1, q=1, workers=4, verbose=True)

    edge_list = n2v.read_edg(filepath, weighted=False, directed=False)
    emd = n2v.embed(dim=128, num_walks=10, walk_length=80, window_size=10, epochs=10)

    n2v_emd = pd.DataFrame(emd, n2v.nodes)

    n2v_emd.columns = ['network_' + str(col) for col in n2v_emd.columns]

    n2v_emd = n2v_emd.reset_index().rename(columns={"index":"string_id"})

    return n2v_emd

ppi_features = Runnode2vec('/itf-fi-ml/shared/users/ziyuzh/svm/data/ppi_clean_2017.txt')

new_columns = ['string_id'] + [f'feature_{i}' for i, col in enumerate(ppi_features.columns) if col != 'string_id']
# Reorder the DataFrame so that 'string_id' is the first column
df_combined = ppi_features[['string_id'] + [col for col in ppi_features.columns if col != 'string_id']]
df_combined.columns = new_columns
df_combined.to_csv('/itf-fi-ml/shared/users/ziyuzh/svm/data/ppi_clean_2017_emb.csv', index=False)

  0%|          | 0/142780 [00:00<?, ?it/s]

# id maps

## ppi id mapping

In [ ]:
import mygene
import pandas as pd
import numpy as np
import pickle

In [32]:
def get_map_df(ensembl_ids,input_type):
    mg = mygene.MyGeneInfo()
    # Query mygene for UniProt and Entrez gene ID mappings
    results = mg.querymany(
        ensembl_ids,
        scopes=input_type,
        fields='uniprot,entrezgene',
        species='human'
    )

    results_df = pd.DataFrame(results)
    results_df = results_df[~results_df['entrezgene'].isna()]
    results_df['uniprot_ids'] = results_df['uniprot'].apply(
        lambda x: list(x.values())[0] if isinstance(x, dict) and 'Swiss-Prot' in x else None)
    results_df = results_df[~results_df['uniprot_ids'].isna()]
    results_df[results_df['uniprot_ids'].apply(lambda x: isinstance(x, list) and len(x) > 1)]
    return results_df


ppi_emb = pd.read_csv('/itf-fi-ml/shared/users/ziyuzh/svm/data/ppi_full_2019_emb.csv')
ppi_ids_map = get_map_df(ppi_emb['string_id'].str.split('.').str[1],'ensembl.protein')

Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
1 input query terms found dup hits:	[('ENSP00000473163', 3)]
1939 input query terms found no hit:	['ENSP00000415070', 'ENSP00000267197', 'ENSP00000415452', 'ENSP00000006101', 'ENSP00000262477', 'ENS


## biocpncept mapping

In [33]:
from gensim.models import KeyedVectors
import os, sys, json, numpy as np

In [34]:
with open('/itf-fi-ml/shared/users/ziyuzh/svm/data/bioconcept/concept_fast.json') as json_file:  
    concept_vectors = json.load(json_file)
print('load', len(concept_vectors), 'concepts')
gene_ids = [key for key, value in concept_vectors.items() if key.startswith('Gene')]
gene_unnest = dict()
all_genes = []
for i in gene_ids:
    all_genes.extend(i.split('_')[1:])
    for j in i.split('_')[1:]:
        gene_unnest[j] = i

load 402712 concepts


In [38]:
bio_ids_map = get_map_df(all_genes,'entrezgene')

Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
6339 input query terms found dup hits:	[('80344', 2), ('2101', 5), ('2099', 23), ('2100', 8), ('10395', 4), ('90627', 3), ('9754', 3), ('92
125631 input query terms found no hit:	['15033', '288593', '108875327', '10201221', '823753', '373705', '829556', '881070', '4179184', '100


## gene2vec

In [40]:
path = '/itf-fi-ml/shared/users/ziyuzh/svm/data/pre_processed_features/expression_emb/gene2vec_dim_200_iter_9.txt'

# Read the data from the text file
with open(path, 'r') as file:
    lines = file.readlines()

# Create a dictionary to store the data
data_dict = {}

# Loop through each line and capture the information in the dictionary
for line in lines:
    parts = line.strip().split('\t')
    key = parts[0]
    values = list(map(float, parts[1].split()))
    data_dict[key] = values

# Convert the dictionary to a DataFrame
gene2vec_df = pd.DataFrame.from_dict(data_dict, orient='index')


In [43]:
gene2vec_ids_map = get_map_df(gene2vec_df.index,'symbol')

Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
721 input query terms found dup hits:	[('PLAC4', 2), ('COPG2IT1', 2), ('DLEU2L', 2), ('COX6CP2', 2), ('GABARAPL3', 2), ('NBPF25P', 2), ('P
4900 input query terms found no hit:	['C6orf226', 'HIST1H2BN', 'H1FX-AS1', 'LOC284014', 'PAPD7', 'LINC01184', 'LOC101929964', 'FAM49B', '


## uniport T5

In [49]:
import h5py

# Path to your .h5 file
file_path = '/itf-fi-ml/shared/users/ziyuzh/svm/data/pre_processed_features/seq_emb/per-protein.h5'

# Open and read file safely
with h5py.File(file_path, 'r') as f:
    ids = list(f.keys())  # list() if you want to use it later

## get interesctions of 2019

In [57]:
ppi_set = set()

for values in ppi_ids_map['uniprot_ids']:
    if isinstance(values, list) and len(values) > 1:
        ppi_set.update(values)  # Add all elements in the list
    else:
        ppi_set.add(values)  # Add the single value


In [59]:
bio_set = set()

for values in bio_ids_map['uniprot_ids']:
    if isinstance(values, list) and len(values) > 1:
        bio_set.update(values)  # Add all elements in the list
    else:
        bio_set.add(values)  # Add the single value

In [61]:
uniport_set = set(ids)

In [ ]:
inter_2019 = uniport_set & bio_set
inter_2019 = inter_2019 & ppi_set
len(inter_2019)

with open('/itf-fi-ml/shared/users/ziyuzh/svm/data/id_maps/inter_2019.pkl', 'wb') as f:
    pickle.dump(inter_2019, f)

15753

## intersection 2017

In [66]:
ppi_emb = pd.read_csv('/itf-fi-ml/shared/users/ziyuzh/svm/data/ppi_full_2016_emb.csv')
ppi_ids_map = get_map_df(ppi_emb['string_id'].str.split('.').str[1],'ensembl.protein')

ppi_set = set()
for values in ppi_ids_map['uniprot_ids']:
    if isinstance(values, list) and len(values) > 1:
        ppi_set.update(values)  # Add all elements in the list
    else:
        ppi_set.add(values)  # Add the single value

gene2vec_set = set()
for values in gene2vec_ids_map['uniprot_ids']:
    if isinstance(values, list) and len(values) > 1:
        gene2vec_set.update(values)  # Add all elements in the list
    else:
        gene2vec_set.add(values)  # Add the single value

inter_2017 = uniport_set & gene2vec_set
inter_2017 = inter_2017 & ppi_set
len(inter_2017)

with open('/itf-fi-ml/shared/users/ziyuzh/svm/data/id_maps/inter_2017.pkl', 'wb') as f:
    pickle.dump(inter_2017, f)

Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
1803 input query terms found no hit:	['ENSP00000006101', 'ENSP00000035383', 'ENSP00000053469', 'ENSP00000205890', 'ENSP00000207437', 'ENS


In [67]:
len(inter_2017)

15397

# run esm.py to get avaliable esm2 features